In [2]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

In [3]:
orders_delivered = orders[
    (orders["order_status"] == 'delivered')
].copy()

len(orders_delivered)

96478

In [4]:
total_orders = orders_delivered.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

len(total_orders)

96478

In [5]:
orders_per_customer = total_orders.groupby("customer_unique_id") ["order_id"].count()

orders_per_customer.value_counts().head(5)

order_id
1    90557
2     2573
3      181
4       28
5        9
Name: count, dtype: int64

In [6]:
(orders_per_customer.value_counts(normalize=True) * 100).round(2).head(5)

order_id
1    97.00
2     2.76
3     0.19
4     0.03
5     0.01
Name: proportion, dtype: float64

In [7]:
order_spend = order_items.groupby("order_id") ["price"].sum().reset_index()

In [8]:
customer_orders_spend = total_orders.merge(
    order_spend,
    on="order_id",
    how="left"
)

customer_value = customer_orders_spend.groupby("customer_unique_id").agg(total_orders=("order_id", "nunique"), total_spend=("price", "sum"))

In [9]:
from numpy.ma import count
customer_value["is_repeat"] = customer_value["total_orders"] > 1

customer_value.groupby("is_repeat") ["total_spend"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
is_repeat,,,
False,90557,137.96,86.96
True,2801,260.05,182.10


In [10]:
total_orders["order_purchase_timestamp"] = pd.to_datetime(total_orders["order_purchase_timestamp"])

repeat_customers_ids = customer_value[customer_value["is_repeat"]].index 

repeat_orders = total_orders[total_orders["customer_unique_id"].isin(repeat_customers_ids)].sort_values(["customer_unique_id", "order_purchase_timestamp"])

In [11]:
repeat_orders["days_since_prev_order"] = (repeat_orders.groupby("customer_unique_id")["order_purchase_timestamp"].diff().dt.days)

In [12]:
repeat_orders["days_since_prev_order"].dropna().agg(["mean", "median"]).round(1)

mean      78.8
median    29.0
Name: days_since_prev_order, dtype: float64